In [20]:
!export HSA_OVERRIDE_GFX_VERSION=10.3.0
!pip install --upgrade pip

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 7.8 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: pip
    Found existing installation: pip 23.0.1
    Uninstalling pip-23.0.1:
      Successfully uninstalled pip-23.0.1


In [34]:
!pip3 install -e .

Obtaining file:///home/mtd/Desktop/Projects/Computer%20Vison/OpticalFlow/memfof_CNNfeaturenet
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for memfof (pyproject.toml) ... done
  Created wheel for memfof: filename=memfof-1.0.3-0.editable-py3-none-any.whl size=6791 sha256=ee9fa784f21fec0b597d53421c577c2d1995a48aae7fd186361eacdea5ffa0df
  Stored in directory: /tmp/pip-ephem-wheel-cache-2f1k24u7/wheels/32/61/f8/8d780d53f0cde728f1afca6614a866d01ebc52459a1f775ec9
Successfully built memfof
  Attempting uninstall: memfof
    Found existing installation: memfof 1.0.3
    Uninstalling memfof-1.0.3:
      Successfully uninstalled memfof-1.0.3


In [35]:
!pip3 install -e .[dev]

Obtaining file:///home/mtd/Desktop/Projects/Computer%20Vison/OpticalFlow/memfof_CNNfeaturenet
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Using cached lightning-2.6.0-py3-none-any.whl.metadata (44 kB)
  Using cached opencv_python_headless-4.12.0.88-cp37-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (19 kB)
  Using cached h5py-3.15.1-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (3.0 kB)
INFO: pip is looking at multiple versions of memfof[dev] to determine which version is compatible with other requirements. This could take a while.
ERROR: Ignored the following yanked versions: 1.11.0, 1.14.0rc1
ERROR: Ignored the following versions that require a different python version: 1.16.0 Requires-Python >=3.11; 1.16.0rc1 Requires-Python >=3.11; 1.16.0rc2 Requires-Python >=3.11; 1.16.1 Requires-

In [ ]:
!pip3 install torch torchvision --extra-index-url https://download.pytorch.org/whl/rocm7.1

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/rocm7.1


In [33]:
!pacman -Q | grep rocm
!python3 -m torch.utils.collect_env

rocm-cmake 7.1.0-1
rocm-core 7.1.0-1
rocm-dbgapi 7.1.0-1
rocm-device-libs 2:7.1.0-1
rocm-gdb 7.1.0-1
rocm-hip-runtime 7.1.0-1
rocm-language-runtime 7.1.0-1
rocm-llvm 2:7.1.0-1
rocm-smi-lib 7.1.0-1
rocminfo 7.1.0-1
/home/mtd/.pyenv/versions/3.10.19/lib/python3.10/runpy.py:126: RuntimeWarning: 'torch.utils.collect_env' found in sys.modules after import of package 'torch.utils', but prior to execution of 'torch.utils.collect_env'; this may result in unpredictable behaviour
  warn(RuntimeWarning(msg))
PyTorch version: 2.9.1+cu128
Is debug build: False
CUDA used to build PyTorch: 12.8
ROCM used to build PyTorch: N/A

OS: EndeavourOS Linux (x86_64)
GCC version: (GCC) 15.2.1 20251112
Clang version: 21.1.6
CMake version: version 4.2.0
Libc version: glibc-2.42

Python version: 3.10.19 (main, Nov 30 2025, 21:53:15) [GCC 15.2.1 20251112] (64-bit runtime)
Python platform: Linux-6.17.9-arch1-1-x86_64-with-glibc2.42
Is CUDA available: False
CUDA runtime version: 13.0.88
CUDA_MODULE_LOADING set to: N

In [32]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} for inference")

if torch.cuda.is_available():
    print("AMD GPU (with ROCm) is available and detected by PyTorch.")
    print(f"Device name: {torch.cuda.get_device_name(0)}")
else:
    print("No AMD GPU with ROCm support detected by PyTorch, or ROCm is not properly installed.")

Using cpu for inference
No AMD GPU with ROCm support detected by PyTorch, or ROCm is not properly installed.


In [1]:
from memfof.model import MEMFOF, AVAILABLE_MODELS

print("List of available models:")
for model_name in AVAILABLE_MODELS:
    print("\t", model_name)

/home/mtd/.pyenv/versions/3.10.19/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


List of available models:
	 MEMFOF-Tartan
	 MEMFOF-Tartan-T
	 MEMFOF-Tartan-T-TSKH
	 MEMFOF-Tartan-T-TSKH-kitti
	 MEMFOF-Tartan-T-TSKH-sintel
	 MEMFOF-Tartan-T-TSKH-spring


In [2]:
model = MEMFOF.from_pretrained("egorchistov/optical-flow-MEMFOF-Tartan-T-TSKH").eval().to(device)

NameError: name 'device' is not defined

In [ ]:
with torch.inference_mode():
    example_input = torch.randint(0, 256, [1, 3, 3, 1080, 1920], device=device)  # [B=1, T=3, C=3, H=1080, W=1920]
    backward_flow, forward_flow = model(example_input)["flow"][-1].unbind(dim=1)  # [B=1, C=2, H=1080, W=1920]

    max_memory_allocated = torch.cuda.max_memory_allocated()
    print(f"Max allocated memory: {max_memory_allocated / 1024**2:.2f} MB")